# 00. Data Loading & Schema Check

## Objective
This notebook validates the SQLite connection, verifies the schema of the `forecasting_dataset` view, and performs high-level checks (data types, shape, and missing values) prior to full Exploratory Data Analysis (EDA).

##  Data Source
* **Dataset:** Walmart Sales Forecast
* **Source:** Kaggle / Hugging Face (`aslanahmedov/walmart-sales-forecast`)
* **Privacy Note:** This dataset is **publicly available** for educational and research purposes. Displays of row samples, column structures, and summary statistics do not expose any proprietary or sensitive information.

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

Database Connection & Catalog Check

In [3]:
DB_PATH = Path("../db/sales.db")
conn = sqlite3.connect(DB_PATH)

# List all tables and views
query_catalog = """
SELECT type, name 
FROM sqlite_master 
WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';
"""

catalog_df = pd.read_sql(query_catalog, conn)
display(catalog_df)

,type,name
0,table,sales
1,table,features
2,table,stores
3,view,forecasting_dataset


Schema and Row Count Analysis

In [4]:
# Analyze each object
for _, row in catalog_df.iterrows():
    object_name = row["name"]
    object_type = row["type"].upper()
    
    count_df = pd.read_sql(f"SELECT COUNT(*) as total FROM {object_name}", conn)
    total_rows = count_df["total"].iloc[0]
    
    schema_df = pd.read_sql(f"PRAGMA table_info({object_name})", conn)
    cols = schema_df["name"].tolist()
    
    print(f"[{object_type}] {object_name}")
    print(f"rows: {total_rows:,}")
    print(f"columns ({len(cols)}): {cols}\n")

conn.close()

[TABLE] sales
rows: 421,570
columns (5): ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']

[TABLE] features
rows: 8,190
columns (12): ['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday']

[TABLE] stores
rows: 45
columns (3): ['Store', 'Type', 'Size']

[VIEW] forecasting_dataset
rows: 421,570
columns (10): ['Store', 'Date', 'Weekly_Sales', 'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Type', 'Size']



Load Forecasting Dataset into Pandas

In [5]:
conn = sqlite3.connect(DB_PATH)

query = "SELECT * FROM forecasting_dataset"

df = pd.read_sql(query, conn)

conn.close()

df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Store,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,CPI,Unemployment,Type,Size
0,1,2010-02-05,24924.50,0,42.31,2.572,211.096358,8.106,A,151315
1,1,2010-02-12,46039.49,1,38.51,2.548,211.242170,8.106,A,151315
2,1,2010-02-19,41595.55,0,39.93,2.514,211.289143,8.106,A,151315
3,1,2010-02-26,19403.54,0,46.63,2.561,211.319643,8.106,A,151315
4,1,2010-03-05,21827.90,0,46.50,2.625,211.350143,8.106,A,151315
